# Optimized Data Cleaning Pipeline
This notebook handles data cleaning, handling missing values, calculating global medians avoiding chunk-median skew, managing outliers using IQR techniques, training regression/classification imputers and performing low-memory chunk processing on the main dataset.

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Config
input_file = '46_Col_final_with_county.parquet'
output_file = 'BER_Cleaned_Optimized.parquet'

## 1. Pre-compute Global Medians
To ensure consistency across chunked data, we load only the necessary efficiency columns into memory to fetch the true mathematical median globally.

In [2]:
print("--- PRE-COMPUTING GLOBAL MEDIANS FOR EFFICIENCY BLOCK ---")
eff_cols = ['HSMainSystemEfficiency', 'WHMainSystemEff']

global_medians = {}
df_eff = pd.read_parquet(input_file, columns=eff_cols)
for col in eff_cols:
    global_medians[col] = df_eff[col].median()
    print(f"Global median for {col}: {global_medians[col]:.2f}")
    
del df_eff # Free memory\n

--- PRE-COMPUTING GLOBAL MEDIANS FOR EFFICIENCY BLOCK ---
Global median for HSMainSystemEfficiency: 90.30
Global median for WHMainSystemEff: 90.30


## 2. Train Iterative Imputation Models
By reading just the training features and the target features, we train models beforehand. \n

*Note*: We suppress regression outliers utilizing the Interquartile Range (IQR) technique before training.

In [4]:
print("\n--- TRAINING BUILDING ENVELOPE IMPUTATION MODELS ---")

train_cols = [
    'Year_of_Construction', 'DwellingTypeDescr', 'StructureType', 
    'VentilationMethod', 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 
    'NoOfSidesSheltered', 'NoOfFansAndVents'
]
df_train = pd.read_parquet(input_file, columns=train_cols)

# Encode building type
dwelling_encoder = LabelEncoder()
df_train['DwellingTypeDescr_enc'] = dwelling_encoder.fit_transform(df_train['DwellingTypeDescr'].astype(str))

feature_cols = ['Year_of_Construction', 'DwellingTypeDescr_enc']

envelope_cols = {
    'StructureType': 'classification',
    'VentilationMethod': 'classification',
    'SuspendedWoodenFloor': 'classification',
    'PercentageDraughtStripped': 'regression',
    'NoOfSidesSheltered': 'regression',
    'NoOfFansAndVents': 'regression',
}

label_encoders = {}
trained_models = {}

for col, task_type in envelope_cols.items():
    print(f"Training model for: {col}... ", end='')
    
    # Extract complete cases for this column to learn from
    mask = df_train[feature_cols + [col]].dropna()
    
    # IQR filtering only for regression to omit noise
    if task_type == 'regression':
        Q1 = mask[col].quantile(0.25)
        Q3 = mask[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR > 0:
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            mask = mask[(mask[col] >= lower_bound) & (mask[col] <= upper_bound)]
            print(f"(IQR Filter: [{lower_bound:.2f}, {upper_bound:.2f}]) ", end='')
            
    X = mask[feature_cols].values
    y = mask[col]
    
    if task_type == 'classification':
        le = LabelEncoder()
        y_enc = le.fit_transform(y.astype(str))
        label_encoders[col] = le
        
        model = LogisticRegression(max_iter=500, random_state=42)
        model.fit(X, y_enc)
        trained_models[col] = ('classification', model)
    else:
        model = LinearRegression()
        model.fit(X, y.values)
        trained_models[col] = ('regression', model)
        
    print("Done!")

del df_train # Free memory
print("Models computational phase finished.")


--- TRAINING BUILDING ENVELOPE IMPUTATION MODELS ---
Training model for: StructureType... Done!
Training model for: VentilationMethod... Done!
Training model for: SuspendedWoodenFloor... Done!
Training model for: PercentageDraughtStripped... (IQR Filter: [82.50, 110.50]) Done!
Training model for: NoOfSidesSheltered... (IQR Filter: [0.50, 4.50]) Done!
Training model for: NoOfFansAndVents... (IQR Filter: [-3.50, 8.50]) Done!
Models computational phase finished.


## 3. Chunked Data Pipeline
Define our transformation logic inside a reusable function. Using NMAR Logical Fills and MAR Regressions based on prior training computations.\n

In [6]:
def process_chunk(chunk, dwelling_encoder, trained_models, label_encoders, global_medians):
    chunk = chunk.copy()
    
    # 0. DROP FEATURES
    # PredominantRoofType is dropped due to >50% missing data and physical driver repetition
    if 'PredominantRoofType' in chunk.columns:
        chunk = chunk.drop(columns=['PredominantRoofType'])

    # 1. SYSTEMS BLOCK (NMAR) -> LOGICAL ZERO FILL
    # Null values reliably indicate absence of supplemental or renewable systems
    zero_fill_num = [
        'HSSupplHeatFraction', 'SHRenewableResources', 'WHRenewableResources', 
        'HSSupplSystemEff', 'WHEffAdjFactor', 'HSEffAdjFactor',
        'SupplSHFuel', 'SupplWHFuel'
    ]
    for col in zero_fill_num:
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna(0)

    # 2. EFFICIENCY BLOCK -> GLOBAL MEDIAN FILL
    # Fill standard efficiency metrics to best generalized expectations
    for col, median_val in global_medians.items():
        if col in chunk.columns:
            chunk[col] = chunk[col].fillna(median_val)
            
    # 3. BUILDING ENVELOPE (MAR) -> REGRESSION FILL
    if 'DwellingTypeDescr' in chunk.columns and 'Year_of_Construction' in chunk.columns:
        chunk['DwellingTypeDescr_enc'] = dwelling_encoder.transform(chunk['DwellingTypeDescr'].astype(str))
        X_chunk = chunk[['Year_of_Construction', 'DwellingTypeDescr_enc']].values
        
        for col, (task_type, model) in trained_models.items():
            if col not in chunk.columns:
                continue
                
            null_mask = chunk[col].isnull()
            if null_mask.sum() == 0:
                continue
                
            X_null = X_chunk[null_mask]
            predicted = model.predict(X_null)
            
            if task_type == 'classification':
                le = label_encoders[col]
                predicted_labels = le.inverse_transform(predicted.astype(int))
                chunk.loc[null_mask, col] = predicted_labels
            else:
                chunk[col] = chunk[col].astype(float) # Pandas v3.0 Series explicit type-casting prevention
                chunk.loc[null_mask, col] = predicted.round(1)
                
        chunk.drop(columns=['DwellingTypeDescr_enc'], inplace=True)
        
    return chunk

## 4. Run Execution\n
Process parquets using streaming iterators limiting memory bloat.\n

In [8]:
print("\n--- STARTING CHUNKED PROCESSING ---")
parquet_file = pq.ParquetFile(input_file)
writer = None

for i in range(parquet_file.num_row_groups):
    print(f"Processing row group {i+1}/{parquet_file.num_row_groups}...")
    chunk = parquet_file.read_row_group(i).to_pandas()
    
    cleaned_chunk = process_chunk(
        chunk, 
        dwelling_encoder, 
        trained_models, 
        label_encoders, 
        global_medians
    )
    
    table = pa.Table.from_pandas(cleaned_chunk)
    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)
    writer.write_table(table)

if writer:
    writer.close()

print(f"\n✅ Done! The optimized, memory-efficient cleaned file has been saved to: {output_file}")


--- STARTING CHUNKED PROCESSING ---
Processing row group 1/2...
Processing row group 2/2...

✅ Done! The optimized, memory-efficient cleaned file has been saved to: BER_Cleaned_Optimized.parquet


## 5. Validation & Summary
After processing, we load the final output to mathematically guarantee our imputation resolved the required data.

In [10]:
import io

print("\n--- FINAL DATASET VALIDATION ---")
# Load the optimized output file
df_clean = pd.read_parquet(output_file)

# Summarize shape
num_rows, num_cols = df_clean.shape
missing_counts = df_clean.isnull().sum()
missing_filtered = missing_counts[missing_counts > 0]

buffer = io.StringIO()
buffer.write("--- FINAL DATASET VALIDATION ---\n")
buffer.write(f"Total Rows: {num_rows:,}\n")
buffer.write(f"Total Columns: {num_cols}\n\n")

buffer.write("--- MISSING VALUES SUMMARY ---\n")
if missing_filtered.empty:
    buffer.write("✅ No missing values found! The engineered imputation was 100% comprehensive.\n")
else:
    buffer.write("⚠️ Outstanding Missing Values:\n")
    buffer.write(missing_filtered.sort_values(ascending=False).to_string())
    buffer.write("\n")

buffer.write("\n--- DATASET INFO ---\n")
info_buf = io.StringIO()
df_clean.info(buf=info_buf)
buffer.write(info_buf.getvalue())

summary_content = buffer.getvalue()
print(summary_content)

# Save to file
summary_file = 'data_summary.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write(summary_content)

print(f"\nValidation summary saved to: {summary_file}")



--- FINAL DATASET VALIDATION ---
--- FINAL DATASET VALIDATION ---
Total Rows: 1,351,582
Total Columns: 45

--- MISSING VALUES SUMMARY ---
✅ No missing values found! The engineered imputation was 100% comprehensive.

--- DATASET INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1351582 entries, 0 to 1351581
Data columns (total 45 columns):
 #   Column                        Non-Null Count    Dtype   
---  ------                        --------------    -----   
 0   CountyName                    1351582 non-null  category
 1   DwellingTypeDescr             1351582 non-null  category
 2   Year_of_Construction          1351582 non-null  float32 
 3   BerRating                     1351582 non-null  float32 
 4   UValueWall                    1351582 non-null  float32 
 5   UValueRoof                    1351582 non-null  float32 
 6   UValueFloor                   1351582 non-null  float32 
 7   UValueWindow                  1351582 non-null  float32 
 8   UvalueDoor              